In [ ]:
import itertools
import networkx as nx

# --- DATA SAMPEL (CONTOH) ---
# Dalam penggunaan nyata, data ini akan dimuat dari hasil simulasi YAFS.

# Daftar node fog dengan properti sumber dayanya
# 'total' adalah kapasitas total, 'used' adalah yang sedang digunakan
daftar_node = [
    {'id': 1, 'ram_total': 20, 'ram_used': 10, 'cpu_total': 1000, 'cpu_used': 500, 'storage_total': 100, 'storage_used': 20},
    {'id': 2, 'ram_total': 10, 'ram_used': 8, 'cpu_total': 800, 'cpu_used': 600, 'storage_total': 50, 'storage_used': 40},
    {'id': 3, 'ram_total': 25, 'ram_used': 10, 'cpu_total': 1500, 'cpu_used': 500, 'storage_total': 200, 'storage_used': 50},
    {'id': 4, 'ram_total': 15, 'ram_used': 15, 'cpu_total': 1000, 'cpu_used': 900, 'storage_total': 100, 'storage_used': 95}
]

# Daftar permintaan (requests) yang telah diproses
# 'is_available' menunjukkan apakah semua layanan dapat dijangkau (untuk metrik Availability)
daftar_permintaan = [
    {'req_id': 101, 'app_id': 'app_A', 'satisfied': True, 'used_existing_placement': True, 'response_time': 2500, 'deadline': 3000, 'requester_node': 1, 'is_available': True},
    {'req_id': 102, 'app_id': 'app_B', 'satisfied': True, 'used_existing_placement': False, 'response_time': 4000, 'deadline': 5000, 'requester_node': 2, 'is_available': True},
    {'req_id': 103, 'app_id': 'app_A', 'satisfied': True, 'used_existing_placement': False, 'response_time': 3100, 'deadline': 3000, 'requester_node': 3, 'is_available': True},
    {'req_id': 104, 'app_id': 'app_C', 'satisfied': False, 'used_existing_placement': False, 'response_time': -1, 'deadline': 4000, 'requester_node': 4, 'is_available': False},
    {'req_id': 105, 'app_id': 'app_B', 'satisfied': True, 'used_existing_placement': True, 'response_time': 3500, 'deadline': 5000, 'requester_node': 1, 'is_available': True},
]

# Matriks penempatan: layanan -> node tempat layanan ditempatkan
placement_matrix = {
    'app_A': {'service_A1': 2, 'service_A2': 3},
    'app_B': {'service_B1': 4},
}

# Representasi topologi jaringan menggunakan NetworkX
# Atribut pada edge: 'pd' (Propagation Delay), 'bw' (Bandwidth)
G = nx.Graph()
G.add_edge(1, 2, pd=3, bw=6e7)
G.add_edge(1, 3, pd=4, bw=6e7)
G.add_edge(2, 3, pd=5, bw=7e7)
G.add_edge(3, 4, pd=3, bw=8e7)


# --- FUNGSI METRIK EVALUASI ---

def calculate_resource_usage(nodes: list) -> dict:
    """
    Menghitung metrik Resource Usage (Penggunaan Sumber Daya).
    Berdasarkan deskripsi dan Eq. 24-27 di paper FSPCN.
    """
    total_ram_used = sum(n['ram_used'] for n in nodes)
    total_ram_total = sum(n['ram_total'] for n in nodes)
    total_cpu_used = sum(n['cpu_used'] for n in nodes)
    total_cpu_total = sum(n['cpu_total'] for n in nodes)
    total_storage_used = sum(n['storage_used'] for n in nodes)
    total_storage_total = sum(n['storage_total'] for n in nodes)

    ram_usage_rate = total_ram_used / total_ram_total if total_ram_total > 0 else 0
    cpu_usage_rate = total_cpu_used / total_cpu_total if total_cpu_total > 0 else 0
    storage_usage_rate = total_storage_used / total_storage_total if total_storage_total > 0 else 0

    # Rata-rata dari ketiga rasio penggunaan
    avg_usage = (ram_usage_rate + cpu_usage_rate + storage_usage_rate) / 3
    
    return {
        'ram_usage_rate': ram_usage_rate,
        'cpu_usage_rate': cpu_usage_rate,
        'storage_usage_rate': storage_usage_rate,
        'average_resource_usage': avg_usage
    }

def calculate_placement_usage(requests: list) -> float:
    """
    Menghitung metrik Placement Usage.
    Berdasarkan Eq. 28 di paper FSPCN.
    """
    srep = sum(1 for r in requests if r['satisfied'] and r['used_existing_placement'])
    sr = sum(1 for r in requests if r['satisfied'])
    
    return srep / sr if sr > 0 else 0.0

def calculate_availability(requests: list) -> float:
    """
    Menghitung metrik Availability.
    Berdasarkan Eq. 29 di paper FSPCN.
    Dalam implementasi ini, diasumsikan 'is_available' sudah dihitung sebelumnya.
    Perhitungan 'is_available' memerlukan pengecekan path dari requester ke semua service.
    """
    total_requests = len(requests)
    if total_requests == 0:
        return 0.0
        
    available_requests = sum(1 for r in requests if r['is_available'])
    
    return available_requests / total_requests

def calculate_meet_deadline(requests: list) -> float:
    """
    Menghitung metrik Meet Deadline.
    Berdasarkan Eq. 30 di paper FSPCN.
    """
    total_requests = len(requests)
    if total_requests == 0:
        return 0.0

    met_deadline_count = sum(1 for r in requests if r['satisfied'] and r['response_time'] <= r['deadline'])

    return met_deadline_count / total_requests

def _calculate_path_delay(path: list, graph: nx.Graph, packet_size: float) -> float:
    """Helper untuk menghitung total delay di sepanjang path."""
    total_delay = 0
    for i in range(len(path) - 1):
        u, v = path[i], path[i+1]
        link_data = graph.get_edge_data(u, v)
        # Formula dari Eq. 31
        link_delay = link_data['pd'] + (packet_size / link_data['bw'])
        total_delay += link_delay
    return total_delay

def calculate_adra(start_node: int, end_node: int, graph: nx.Graph, packet_size: float) -> float:
    """
    Menghitung Average Delay from Requester to Application (ADRA).
    Berdasarkan Algoritma 4 di paper FSPCN.
    """
    try:
        path = nx.shortest_path(graph, source=start_node, target=end_node)
        return _calculate_path_delay(path, graph, packet_size)
    except nx.NetworkXNoPath:
        return float('inf') # Tidak ada jalur

def calculate_adsa(app_id: str, placement: dict, graph: nx.Graph, packet_size: float) -> float:
    """
    Menghitung Average Delay between Services of an Application (ADSA).
    Berdasarkan Algoritma 5 di paper FSPCN.
    """
    if app_id not in placement or len(placement[app_id]) < 2:
        return 0.0 # Tidak ada delay antar layanan jika hanya ada 1 atau 0 layanan
    
    # Dapatkan daftar node di mana layanan aplikasi ini ditempatkan
    service_nodes = list(set(placement[app_id].values()))
    
    total_inter_service_delay = 0
    pair_count = 0
    
    # Hitung delay untuk setiap pasangan node layanan
    for pair in itertools.combinations(service_nodes, 2):
        start_node, end_node = pair
        # Paper (Alg. 5 line 7) menyebutkan penggunaan ADRA untuk menghitung delay antar service
        delay = calculate_adra(start_node, end_node, graph, packet_size)
        if delay != float('inf'):
            total_inter_service_delay += delay
            pair_count += 1
            
    return total_inter_service_delay / pair_count if pair_count > 0 else 0.0


# --- CONTOH PENGGUNAAN ---
if __name__ == "__main__":
    print("--- Menghitung Metrik Evaluasi Kinerja ---")
    
    # 1. Resource Usage
    resource_metrics = calculate_resource_usage(daftar_node)
    print("\n1. Metrik Resource Usage:")
    print(f"   - Rata-rata Penggunaan Sumber Daya (µ): {resource_metrics['average_resource_usage']:.2%}")
    
    # 2. Placement Usage
    placement_usage_metric = calculate_placement_usage(daftar_permintaan)
    print(f"\n2. Metrik Placement Usage (ε): {placement_usage_metric:.2%}")
    
    # 3. Availability
    availability_metric = calculate_availability(daftar_permintaan)
    print(f"\n3. Metrik Availability (φ): {availability_metric:.2%}")

    # 4. Meet Deadline
    meet_deadline_metric = calculate_meet_deadline(daftar_permintaan)
    print(f"\n4. Metrik Meet Deadline: {meet_deadline_metric:.2%}")
    
    # 5. Delay
    # Ukuran paket default untuk perhitungan delay
    DEFAULT_PACKET_SIZE = 2_000_000 # dalam bytes

    # Contoh menghitung ADRA untuk satu permintaan
    req_contoh = daftar_permintaan[0]
    app_contoh_id = req_contoh['app_id']
    entry_point_node = placement_matrix[app_contoh_id]['service_A1']
    adra_metric = calculate_adra(req_contoh['requester_node'], entry_point_node, G, DEFAULT_PACKET_SIZE)
    
    # Contoh menghitung ADSA untuk satu aplikasi
    adsa_metric = calculate_adsa(app_contoh_id, placement_matrix, G, DEFAULT_PACKET_SIZE)
    
    print("\n5. Metrik Delay (contoh untuk app_A):")
    print(f"   - ADRA (Requester ke App Entry Point): {adra_metric:.2f} ms")
    print(f"   - ADSA (Antar Layanan Aplikasi): {adsa_metric:.2f} ms")
    print(f"   - Rata-rata Delay Total (ADRA+ADSA): {(adra_metric + adsa_metric):.2f} ms")

--- Menghitung Metrik Evaluasi Kinerja ---

1. Metrik Resource Usage:
   - Rata-rata Penggunaan Sumber Daya (µ): 55.04%

2. Metrik Placement Usage (ε): 50.00%

3. Metrik Availability (φ): 80.00%

4. Metrik Meet Deadline: 60.00%

5. Metrik Delay (contoh untuk app_A):
   - ADRA (Requester ke App Entry Point): 3.03 ms
   - ADSA (Antar Layanan Aplikasi): 5.03 ms
   - Rata-rata Delay Total (ADRA+ADSA): 8.06 ms
